# Tugas Manipulasi Data Pandas

Pandas adalah salah satu library Python yang digunakan untuk mengolah, membersihkan, memanipulasi, dan menganalisis data dalam bentuk tabel.

Library Pandas memudahkan proses pengolahan data seperti membaca file CSV, memperbaiki data yang kosong, mengubah tipe data, memilih kolom tertentu, hingga melakukan perhitungan.

## Task 1 - Indexing & Selection (Pembersihan Anomali & Imputasi)

In [1]:
import pandas as pd

ds = pd.read_csv('/content/paket_a_transaksi_pos_2024_2025.csv')

**1. Pembersihan Format Harga**

Kolom `harga_satuan` masih bertipe `object` karena menggunakan koma sebagai pemisah desimal. Mengubah koma menjadi titik, kemudian konversikan menjadi bertipe data `float`.

In [2]:
print(ds['harga_satuan'].dtype)

object


In [3]:
ds['harga_satuan'] = pd.to_numeric(
    ds['harga_satuan'].astype(str).str.replace(',', '.', regex=False)
).astype(float)

print(ds['harga_satuan'].dtype)

float64


**2. Pembersihan Kuantitas Negatif**

Terdapat data yang jumlahnya kurang dari 0, gunakan boolean indexing dan `.loc[]` untuk mencari nilai kurang dari 0. Kemudian ubah menjadi nilai positif menggunakan `.abs()`.

In [4]:
print("Jumlah data negatif:", (ds['jumlah'] < 0).sum())

Jumlah data negatif: 41


In [5]:
ds.loc[ds['jumlah'] < 0, 'jumlah'] = ds.loc[
    ds['jumlah'] < 0, 'jumlah'
].abs()

print("Jumlah data negatif:", (ds['jumlah'] < 0).sum())

Jumlah data negatif: 0


**3. Mengisi Nilai Kosong Kasir**

Terdapat nilai kosong pada `kode_kasir`, gunakan `.loc[]` untuk mengisi nilai kosong dengan label `K-UNKNOWN`.

In [6]:
print("Jumlah nilai kosong:", ds['kode_kasir'].isna().sum())

Jumlah nilai kosong: 261


In [7]:
ds.loc[ds['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'

print("Jumlah nilai kosong:", ds['kode_kasir'].isna().sum())

Jumlah nilai kosong: 0


**4. Seleksi Posisi**

Melakukan seleksi menggunakan `.iloc[]`, dengan tujuan menampilkan kolom `id_transaksi`, `tanggal`, `kode_gerai` dan `jumlah` untuk 5 baris pertama.

In [8]:
ds.iloc[:5, [0, 1, 2, 5]]

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


## Task 2 - Operasi Aritmatika (Feature Engineering)

**1. Perhitungan Omzet Kotor (total_kotor)**

Menghitung omzet sebelum diskon dengan mengalikan harga satuan dan jumlah barang.

---
`Total_Kotor = harga_satuan * jumlah`




In [9]:
ds['total_kotor'] = ds['harga_satuan'] * ds['jumlah']

display(ds['total_kotor'].head())

,total_kotor
0,52000.0
1,21000.0
2,22500.0
3,13000.0
4,6000.0


**2. Perhitungan Nominal Diskon (nominal_diskon)**

Menghitung jumlah potongan harga berdasarkan persentase diskon.

---
`nominal_diskon = total_kotor * (diskon_persen / 100)`

In [10]:
ds['nominal_diskon'] = ds['total_kotor'] * (ds['diskon_persen'] / 100)

display(ds['nominal_diskon'].tail())

,nominal_diskon
5025,0.0
5026,0.0
5027,0.0
5028,4350.0
5029,0.0


**3. Perhitungan Omzet Bersih dan Pajak (total_bersih)**

Menghitung omzet setelah diikurangi diskon dan ditambah PPN sebesar 11%.

---
`total_bersih = (total_kotor - nominal_diskon) * 1.11`

In [11]:
ds['total_bersih'] = (
    ds['total_kotor'] - ds['nominal_diskon']
) * 1.11

display(ds['total_bersih'].head())

,total_bersih
0,57720.0
1,23310.0
2,24975.0
3,14430.0
4,6660.0


## Task 3 - Filtering (Penyaringan Data)

**1. Filter Multi Kondisi**

Menampilkan transaksi yang memenuhi semua kriteria :
* `status` bernilai `SELESAI`.
* `kode_gerai` adalah `G05` atau `G09`.
* `total_bersih` lebih dari Rp100.000


Menggunakan Boolean Indexing dengan operator :
1. `&` berarti DAN (semua kondisi terpenuhi).
2. `|` berarti ATAU (salah satu kondisi terpenuhi).
3. Setiap kondisi perlu diberi tanda kurung.




In [12]:
filter_transaksi = ds[
    (ds['status'] == 'SELESAI') &
    ((ds['kode_gerai'] == 'G05') | (ds['kode_gerai'] == 'G09')) &
    (ds['total_bersih'] > 100000)
]

display(filter_transaksi)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,total_kotor,nominal_diskon,total_bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,47500.0,10,SELESAI,190000.0,19000.0,189810.0
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,30500.0,0,SELESAI,122000.0,0.0,135420.0
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,28500.0,0,SELESAI,114000.0,0.0,126540.0
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,46500.0,0,SELESAI,139500.0,0.0,154845.0
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,20500.0,0,SELESAI,123000.0,0.0,136530.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4877,TRX0243898,2024-04-23,G05,K008,SGR1204,12,51000.0,0,SELESAI,612000.0,0.0,679320.0
4911,TRX0241966,2024-04-22,G09,K024,SGR1206,3,42000.0,0,SELESAI,126000.0,0.0,139860.0
4925,TRX0240377,2025-06-11,G09,K028,XXX9969,12,51500.0,15,SELESAI,618000.0,92700.0,583083.0
4964,TRX0243219,2025-11-17,G09,K003,SGR1204,6,43500.0,0,SELESAI,261000.0,0.0,289710.0


In [13]:
print("Jumlah data yang memenuhi kriteria:", len(filter_transaksi))

Jumlah data yang memenuhi kriteria: 139


**2. Filter Menggunakan `.query()`**

Menampilkan traksaksi dari gerai `G10` yang memiliki jumnlah pembelian minimal 5 unit.

In [14]:
filter_g10 = ds.query("kode_gerai == 'G10' and jumlah >= 5")

display(filter_g10)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,total_kotor,nominal_diskon,total_bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,45500.0,0,SELESAI,546000.0,0.0,606060.0
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,13000.0,0,SELESAI,78000.0,0.0,86580.0
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,26000.0,0,SELESAI,312000.0,0.0,346320.0
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,5500.0,5,SELESAI,66000.0,3300.0,69597.0
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,47000.0,0,SELESAI,282000.0,0.0,313020.0
964,TRX0244939,2025-07-08,G10,K009,SGR1200,6,52000.0,10,SELESAI,312000.0,31200.0,311688.0
989,TRX0243076,2025-02-18,G10,K003,MKN1100,6,18000.0,0,SELESAI,108000.0,0.0,119880.0
1023,TRX0242437,2025-09-18,G10,K030,MKN1104,6,6000.0,0,SELESAI,36000.0,0.0,39960.0
1260,TRX0240537,2025-05-09,G10,K027,BYI1509,6,54000.0,0,SELESAI,324000.0,0.0,359640.0
1296,TRX0240698,2024-01-24,G10,K022,MKN1109,6,29000.0,0,SELESAI,174000.0,0.0,193140.0


## Task 4 - Sorting (Pengurutan Data)

**1. Multi-Column Sorting**

Mengurutkan DataFrame berdasarkan dua kelompok sekaligus:
1. `kode_gerai` secara alfabetis naik (A-Z).
2. `total_bersih` dari nilai tertinggi ke terendah (descending).

In [15]:
ds_sorted = ds.sort_values(
    by=['kode_gerai', 'total_bersih'],
    ascending=[True, False]
)

display(ds_sorted)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,total_kotor,nominal_diskon,total_bersih
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.00,4.202016e+07
2956,TRX0241367,2025-05-15,G01,K006,RMT1408,12,50500.0,0,SELESAI,606000.0,0.00,6.726600e+05
3588,TRX0240608,2025-07-25,G01,K006,MKN1101,12,47000.0,0,SELESAI,564000.0,0.00,6.260400e+05
2937,TRX0240631,2024-12-24,G01,K013,RMT1408,12,42000.0,0,SELESAI,504000.0,0.00,5.594400e+05
5,TRX0240658,2025-07-18,G01,K008,RMT1408,12,45500.0,10,SELESAI,546000.0,54600.00,5.454540e+05
...,...,...,...,...,...,...,...,...,...,...,...,...
2291,TRX0244124,2024-01-26,G12,K035,MIN1008,1,5500.0,20,SELESAI,5500.0,1100.00,4.884000e+03
3850,TRX0242420,2024-01-29,G12,K019,RMT1406,1,4000.0,0,SELESAI,4000.0,0.00,4.440000e+03
4228,TRX0242364,2025-03-22,G12,K032,PRW1300,1,4500.0,20,SELESAI,4500.0,900.00,3.996000e+03
3919,TRX0244437,2025-09-28,G12,K015,MKN1100,3,16.5,10,SELESAI,49.5,4.95,4.945050e+01


**2. Top-N Records**

Mengambil 5 transaksi dengan nilai`total_bersih` paling tinggi dari seluruh dataset menggunakan fungsi `.nlargest()`.

In [16]:
top_transaksi = ds.nlargest(5, 'total_bersih')

display(top_transaksi)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,total_kotor,nominal_diskon,total_bersih
994,TRX0241002,2024-03-18,G11,K038,MKN1101,2093,48000.0,10,SELESAI,100464000.0,10046400.0,100363536.0
4985,TRX0244221,2024-12-09,G09,K005,BYI1500,1301,51500.0,10,SELESAI,67001500.0,6700150.0,66934498.5
467,TRX0243696,2024-02-27,G06,K_UNKNOWN,SGR1201,892,54000.0,0,SELESAI,48168000.0,0.0,53466480.0
3393,TRX0241029,2025-05-12,G07,K028,SGR1205,1983,23000.0,10,SELESAI,45609000.0,4560900.0,45563391.0
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.0,42020160.0


## Task 5 - Grouping * Agregasi (Pelaporan Kinerja Gerai)

**1. Laporan Agregasi Per Gerai**

Membuat tabel ringkasan bernama `laporan_gerai` yang hanya menggunakan transaksi dengan status `SELESAI`.

Metrik yang dihitung:
| Metrik | Operasi | Keterangan |
|---|---|---|
| jumlah_transaksi | count | Menghitung banyaknya ID transaksi |
| total_unit_terjual | sum | Menjumlahkan seluruh barang terjual |
| total_omzet_bersih | sum | Menjumlahkan omzet bersih |
| rata_rata_transaksi | mean | Menghitung rata-rata omzet bersih per transaksi |

In [17]:
#Filter Transaksi Yang Selesai
ds_selesai = ds[ds['status'] == 'SELESAI']

In [18]:
#Grouping dan Agregasi
laporan_gerai = ds_selesai.groupby('kode_gerai').agg(
    jumlah_transaksi=('id_transaksi', 'count'),
    total_unit_terjual=('jumlah', 'sum'),
    total_omzet_bersih=('total_bersih', 'sum'),
    rata_rata_transaksi=('total_bersih', 'mean')
)

In [19]:
laporan_gerai = laporan_gerai.sort_values(
    by='total_omzet_bersih',
    ascending=False
).round(2).reset_index()

pd.options.display.float_format = '{:,.2f}'.format

display(laporan_gerai)

,kode_gerai,jumlah_transaksi,total_unit_terjual,total_omzet_bersih,rata_rata_transaksi
0,G11,398,3170,"125,353,611.58","314,958.82"
1,G06,383,3103,"91,139,358.86","237,961.77"
2,G09,379,3027,"90,196,058.29","237,984.32"
3,G07,384,4284,"80,361,811.64","209,275.55"
4,G01,450,2635,"70,779,587.73","157,287.97"
5,G03,372,3274,"59,809,701.21","160,778.77"
6,G10,401,1742,"58,922,477.71","146,938.85"
7,G08,408,1700,"35,502,561.90","87,016.08"
8,G12,406,1010,"24,644,829.45","60,701.55"
9,G04,399,1042,"24,535,765.12","61,493.15"


**2. Transformasi Kelompok (Window Function)**

Menambahkan kolom baru bernama `kontribusi_gerai_persen`.
Kolom baru ini menunjukkan persentase kontribusi omzet bersih setiap transaksi terhadap total omzet bersih gerainya.

---
`kontribusi (%) = (total_bersih / total_omzet_bersih) * 100`

In [20]:
ds['kontribusi_gerai_persen'] = (
    ds['total_bersih'] /
    ds.groupby('kode_gerai')['total_bersih'].transform('sum')
) * 100

display(ds[['kontribusi_gerai_persen']].head())

,kontribusi_gerai_persen
0,0.06
1,0.03
2,0.02
3,0.06
4,0.03
